In [0]:
CREATE OR REFRESH MATERIALIZED VIEW targets AS
SELECT ae.ae_email, t.Region_Level_1, t.Region_Level_2, t.Region_Level_3, t.user_id, t.dollars as fin_target, t.fiscal_year, concat("FY'", right(t.fiscal_year, 2), ' Q', t.fiscal_quarter) as fiscal_quarter, cast(t.fiscal_quarter as int) as fiscal_quarter_number
FROM main.gtm_silver.targets_individual t
INNER JOIN ae_list ae ON t.Email = ae.ae_email
where t.Business_Unit = '${business_unit}'
and t.Region_Level_1 = '${region_level_1}'
and t.type_target = 'dbu'
and t.snapshot_date = (select max(snapshot_date) from main.gtm_silver.targets_individual)

In [0]:
CREATE OR REFRESH MATERIALIZED VIEW account_targets AS
SELECT
  t.account_id, t.account_name, ae.ae_email, ih.Region_Level_1, ih.Region_Level_2, ih.Region_Level_3, ih.user_id, t.dbu_dollar_target as fin_target, t.fiscal_year, t.fiscal_quarter, cast(right(t.fiscal_quarter, 1) as int) as fiscal_quarter_number
FROM main.gtm_silver.targets_account AS t
INNER JOIN main.gtm_silver.account_dim ad
  ON t.account_id = ad.account_id
INNER JOIN main.gtm_silver.individual_hierarchy_salesforce ih
  ON ad.account_executive_user_id = ih.user_id
INNER JOIN ae_list ae ON ih.user_id = ae.user_id 
WHERE ih.Business_Unit = '${business_unit}'
  AND ih.Region_Level_1 = '${region_level_1}'

In [0]:
CREATE OR REFRESH MATERIALIZED VIEW ds_forecast_account AS
-- Rolls up forecast_ds across all accounts under each AE/manager's org, matching how the
-- 'forecast_actuals' table rolls up dbu_actuals via concatenated_emails (direct account_executive_user_id
-- matching only covers individual AEs and returns 0 for managers who own no accounts directly).
select 
  ae.user_id,
  d.account_id,
  d.fiscal_quarter_end_date,
  d.month_end_date as fiscal_month_end_date,
  sum(d.forecast_ds) as current_ds_forecast
from ae_list ae
inner join main.gtm_silver.forecast_consumption_ds_account d
  on d.concatenated_emails like '%' || ae.ae_email || '%'
where d.date_grain = 'Fiscal Quarter'
and d.business_unit = '${business_unit}'
and d.region_level_1 = '${region_level_1}'
and d.snapshot_date = (select max(snapshot_date) from main.gtm_silver.forecast_consumption_ds_account where business_unit = '${business_unit}' and region_level_1 = '${region_level_1}')
group by all

In [0]:
CREATE OR REFRESH MATERIALIZED VIEW ds_forecast_ae AS
select user_id,  
fiscal_quarter_end_date,
sum(current_ds_forecast) as current_ds_forecast
from ds_forecast_account
group by all

In [0]:
CREATE OR REFRESH MATERIALIZED VIEW sales_forecast AS
select ae.ae_email, f.forecast_owner_id as user_id, f.fiscal_quarter_end_date
  , coalesce(f.submitted_my_call, 0) as submitted_my_call
  , 0 as prev_submitted_my_call
  , coalesce(f.submitted_my_call_w_closed_month_actuals, 0) as submitted_my_call_w_closed_month_actuals
  , coalesce(f.submitted_direct_field_consumption_forecast, 0) as submitted_direct_field_consumption_forecast
  , coalesce(dsf.current_ds_forecast, 0) as current_ds_forecast
  , coalesce(f.submitted_weighted_projection, 0) as submitted_weighted_projection
  , ae.bu_plus_2_lead, ae.bu_plus_3_lead, ae.sales_level, ae.user_name
from main.gtm_silver.forecast_consumption_mcp_individual as f
inner join ae_list ae on f.Email = ae.ae_email
left join ds_forecast_ae dsf
  on dsf.user_id = ae.user_id
  and dsf.fiscal_quarter_end_date = f.fiscal_quarter_end_date
where f.Business_Unit = '${business_unit}'
and f.Region_Level_1 = '${region_level_1}'
and f.snapshot_date = (select max(snapshot_date) from main.gtm_silver.forecast_consumption_mcp_individual where Business_Unit = '${business_unit}' and Region_Level_1 = '${region_level_1}')

In [0]:
CREATE OR REFRESH MATERIALIZED VIEW account_forecast_cte AS
select 
  f.account_id, a.account_name, i.Email as ae_email, f.account_executive_user_id as user_id, 
  f.forecast_fiscal_quarter_end_date as fiscal_quarter_end_date
  , ae.bu_plus_2_lead, ae.bu_plus_3_lead, ae.sales_level, ae.user_name
  , coalesce(sum(f.submitted_ae_forecast), 0) as submitted_my_call
  , 0 as prev_submitted_my_call
  , coalesce(sum(f.submitted_ae_forecast), 0) as submitted_direct_field_consumption_forecast
  , coalesce(sum(f.submitted_ae_forecast_w_closed_month_actuals), 0) as submitted_my_call_w_closed_month_actuals
  , coalesce(max(dsc.current_ds_forecast), 0) as current_ds_forecast --using max because forecast_consumption_mcp_account is at month level whilst ds_forecast_account at quarter level - this is to avoid duplication.
  , coalesce(sum(f.submitted_weighted_projection), 0) as submitted_weighted_projection
from main.gtm_silver.forecast_consumption_mcp_account as f
inner join ae_list ae on f.account_executive_user_id = ae.user_id
left join main.gtm_silver.account_dim as a
  on f.account_id = a.account_id
inner join main.gtm_silver.individual_hierarchy_salesforce as i
  on f.account_executive_user_id = i.user_id  
left join ds_forecast_account dsc
  on dsc.account_id = f.account_id
  and dsc.user_id = ae.user_id
  and dsc.fiscal_quarter_end_date = f.forecast_fiscal_quarter_end_date
where i.Business_Unit = '${business_unit}'
and i.Region_Level_1 = '${region_level_1}'
and f.snapshot_date = (select max(snapshot_date) from main.gtm_silver.forecast_consumption_mcp_account)
group by all

In [0]:
CREATE OR REFRESH MATERIALIZED VIEW actuals AS
select ae.user_id, ae.ae_email, c.fiscal_quarter_start_date
  , ae.bu_plus_2_lead, ae.bu_plus_3_lead
  , ae.bu_plus_1_lead_email, ae.bu_plus_2_lead_email, ae.bu_plus_3_lead_email, ae.sales_level, ae.user_name
  , ae.business_unit AS Business_Unit, ae.region_level_1 AS Region_Level_1, ae.region_level_2 AS Region_Level_2, ae.region_level_3 AS Region_Level_3
  , sum(c.dbu_dollars_qtd) as dbu_actuals
  , sum(c.dbu_dollars_t7d_avg) as dbu_dollars_t7d_avg, sum(c.dbu_dollars_t28d_avg) as dbu_dollars_t28d_avg
  , sum(c.dbu_dollars_t7d_avg_prev) as dbu_dollars_t7d_avg_prev, sum(c.dbu_dollars_t28d_avg_prev) as dbu_dollars_t28d_avg_prev
from ae_list ae
inner join main.gtm_gold.materialized__view_account_obt as c
  on c.concatenated_emails like '%' || ae.ae_email || '%' -- I need the Sales hierarchy, not just the AE
where c.business_unit = '${business_unit}'
and c.subregion_level_1 = '${region_level_1}'
group by all

In [0]:
CREATE OR REFRESH MATERIALIZED VIEW account_actuals AS
select ae.user_id, ae.ae_email
  , ae.bu_plus_2_lead, ae.bu_plus_3_lead
  , ae.bu_plus_1_lead_email, ae.bu_plus_2_lead_email, ae.bu_plus_3_lead_email, ae.sales_level, ae.user_name
  , ar.business_unit AS Business_Unit, ar.subregion_level_1 AS Region_Level_1, ar.subregion_level_2 AS Region_Level_2, ar.subregion_level_3 AS Region_Level_3
  , c.account_id, c.account_name, c.fiscal_quarter_start_date
  , coalesce(sum(c.dbu_dollars_qtd), 0) as dbu_actuals
  , coalesce(sum(c.dbu_dollars_t7d_avg), 0) as dbu_dollars_t7d_avg
  , coalesce(sum(c.dbu_dollars_t28d_avg), 0) as dbu_dollars_t28d_avg
  , coalesce(sum(c.dbu_dollars_t7d_avg_prev), 0) as dbu_dollars_t7d_avg_prev
  , coalesce(sum(c.dbu_dollars_t28d_avg_prev), 0) as dbu_dollars_t28d_avg_prev
from ae_list ae
inner join main.gtm_gold.materialized__view_account_obt as c
  on c.account_executive_user_id = ae.user_id
  --on c.concatenated_emails like '%' || ae.ae_email || '%' --
left outer join account_region as ar
  on ar.account_id = c.account_id
where c.business_unit = '${business_unit}'
and c.subregion_level_1 = '${region_level_1}'
group by all

In [0]:
CREATE OR REFRESH MATERIALIZED VIEW target_forecast_actuals AS
select a.user_id, a.ae_email, a.bu_plus_2_lead, a.bu_plus_3_lead, a.bu_plus_1_lead_email, a.bu_plus_2_lead_email, a.bu_plus_3_lead_email, a.sales_level, a.user_name
  , a.Business_Unit, a.Region_Level_1, a.Region_Level_2, a.Region_Level_3
  , d.fy, d.q, d.fq, a.fiscal_quarter_start_date, a.dbu_actuals
  , a.dbu_dollars_t7d_avg, a.dbu_dollars_t28d_avg, a.dbu_dollars_t7d_avg_prev, a.dbu_dollars_t28d_avg_prev
  , t.fin_target
  , case when d.is_quarter_closed then a.dbu_actuals else f.submitted_my_call end as submitted_my_call
  , case when d.is_quarter_closed then a.dbu_actuals else f.submitted_direct_field_consumption_forecast end as submitted_direct_field_consumption_forecast
  , f.current_ds_forecast, f.submitted_weighted_projection
  , coalesce(try_divide(f.submitted_my_call - f.prev_submitted_my_call, nullif(f.prev_submitted_my_call, 0)), 0) as forecast_change_pct
  , coalesce(a.dbu_actuals, 0) as dbu_actuals_coalesced
  , coalesce(case when d.is_quarter_closed then a.dbu_actuals else submitted_my_call end, 0) as dbu_actuals_or_forecast
  , first_value(a.dbu_dollars_t7d_avg) over(partition by a.ae_email order by d.is_current_fiscal_quarter desc) AS dbu_dollars_t7d_avg_latest
  , coalesce(nullif(a.dbu_dollars_t7d_avg, 0), dbu_dollars_t7d_avg_latest) as dbu_dollars_t7d_adj --for future quarters, use the latest t7d available.
  , first_value(a.dbu_dollars_t28d_avg) over(partition by a.ae_email order by d.is_current_fiscal_quarter desc) AS dbu_dollars_t28d_avg_latest
  , coalesce(nullif(a.dbu_dollars_t28d_avg, 0), dbu_dollars_t28d_avg_latest) as dbu_dollars_t28d_adj --for future quarters, use the latest t28d available.
  , first_value(a.dbu_dollars_t7d_avg_prev) over(partition by a.ae_email order by d.is_current_fiscal_quarter desc) AS dbu_dollars_t7d_avg_prev_latest
  , coalesce(nullif(a.dbu_dollars_t7d_avg_prev, 0), dbu_dollars_t7d_avg_prev_latest) as dbu_dollars_t7d_prev_adj --for future quarters, use the latest t7d_prev available.
  , first_value(a.dbu_dollars_t28d_avg_prev) over(partition by a.ae_email order by d.is_current_fiscal_quarter desc) AS dbu_dollars_t28d_avg_prev_latest
  , coalesce(nullif(a.dbu_dollars_t28d_avg_prev, 0), dbu_dollars_t28d_avg_prev_latest) as dbu_dollars_t28d_prev_adj --for future quarters, use the latest t28d_prev available.
  , case when d.is_quarter_closed then 0 else dbu_actuals_coalesced end dbu_actuals_current_quarter
  , case when d.is_quarter_closed then 0 else (dbu_dollars_t7d_adj * d.days_left_in_quarter) end as t7d_proj_left_in_quarter
  , case when d.is_quarter_closed then 0 else (dbu_dollars_t28d_adj * d.days_left_in_quarter) end as t28d_proj_left_in_quarter
  , coalesce(try_divide(f.submitted_my_call - dbu_actuals_coalesced, d.days_left_in_quarter), 0) as target_t7d  
from actuals as a
inner join financial_quarters d
on d.fiscal_quarter_start_date = a.fiscal_quarter_start_date
left outer join sales_forecast as f 
on f.fiscal_quarter_end_date = d.fiscal_quarter_end_date and f.ae_email = a.ae_email
left outer join targets as t 
on t.fiscal_year = d.fy and t.fiscal_quarter_number = d.q and t.user_id = a.user_id

In [0]:
CREATE OR REFRESH MATERIALIZED VIEW account_target_forecast_actuals AS
select a.user_id, a.bu_plus_2_lead, a.bu_plus_3_lead, a.bu_plus_1_lead_email, a.bu_plus_2_lead_email, a.bu_plus_3_lead_email, a.sales_level, a.user_name
  , a.Business_Unit, a.Region_Level_1, a.Region_Level_2, a.Region_Level_3
  , a.ae_email, a.account_id, a.account_name, a.fiscal_quarter_start_date, d.fy, d.q, d.fq
  , coalesce(t.fin_target, 0) as fin_target
  , case when d.is_quarter_closed then coalesce(a.dbu_actuals, 0) else coalesce(f.submitted_my_call, 0) end as submitted_my_call
  , case when d.is_quarter_closed then coalesce(a.dbu_actuals, 0) else coalesce(f.submitted_direct_field_consumption_forecast, 0) end as submitted_direct_field_consumption_forecast
  , coalesce(f.current_ds_forecast, 0) as current_ds_forecast
  , coalesce(f.submitted_weighted_projection, 0) as submitted_weighted_projection
  , coalesce(try_divide(f.submitted_my_call - f.prev_submitted_my_call, nullif(f.prev_submitted_my_call, 0)), 0) as forecast_change_pct
  , coalesce(a.dbu_actuals, 0) as dbu_actuals
  , coalesce(a.dbu_dollars_t7d_avg, 0) as dbu_dollars_t7d_avg
  , coalesce(a.dbu_dollars_t28d_avg, 0) as dbu_dollars_t28d_avg
  , coalesce(a.dbu_dollars_t7d_avg_prev, 0) as dbu_dollars_t7d_avg_prev
  , coalesce(a.dbu_dollars_t28d_avg_prev, 0) as dbu_dollars_t28d_avg_prev    
  , coalesce(a.dbu_actuals, 0) as dbu_actuals_coalesced
  , coalesce(case when d.is_quarter_closed then a.dbu_actuals else f.submitted_my_call end, 0) as dbu_actuals_or_forecast
  , first_value(a.dbu_dollars_t7d_avg) over(partition by a.ae_email, a.account_id order by d.is_current_fiscal_quarter desc) AS dbu_dollars_t7d_avg_latest
  , coalesce(nullif(a.dbu_dollars_t7d_avg, 0), dbu_dollars_t7d_avg_latest) as dbu_dollars_t7d_adj
  , first_value(a.dbu_dollars_t28d_avg) over(partition by a.ae_email, a.account_id order by d.is_current_fiscal_quarter desc) AS dbu_dollars_t28d_avg_latest
  , coalesce(nullif(a.dbu_dollars_t28d_avg, 0), dbu_dollars_t28d_avg_latest) as dbu_dollars_t28d_adj
  , first_value(a.dbu_dollars_t7d_avg_prev) over(partition by a.ae_email, a.account_id order by d.is_current_fiscal_quarter desc) AS dbu_dollars_t7d_avg_prev_latest
  , coalesce(nullif(a.dbu_dollars_t7d_avg_prev, 0), dbu_dollars_t7d_avg_prev_latest) as dbu_dollars_t7d_prev_adj
  , first_value(a.dbu_dollars_t28d_avg_prev) over(partition by a.ae_email, a.account_id order by d.is_current_fiscal_quarter desc) AS dbu_dollars_t28d_avg_prev_latest
  , coalesce(nullif(a.dbu_dollars_t28d_avg_prev, 0), dbu_dollars_t28d_avg_prev_latest) as dbu_dollars_t28d_prev_adj
  , case when d.is_quarter_closed then 0 else dbu_actuals_coalesced end dbu_actuals_current_quarter
  , case when d.is_quarter_closed then 0 else (dbu_dollars_t7d_adj * d.days_left_in_quarter) end as t7d_proj_left_in_quarter
  , case when d.is_quarter_closed then 0 else (dbu_dollars_t28d_adj * d.days_left_in_quarter) end as t28d_proj_left_in_quarter
  , coalesce(try_divide(f.submitted_my_call - dbu_actuals_coalesced, d.days_left_in_quarter), 0) as target_t7d
from account_actuals as a
inner join financial_quarters d
  on d.fiscal_quarter_start_date = a.fiscal_quarter_start_date
left outer join account_forecast_cte as f
  on f.fiscal_quarter_end_date = d.fiscal_quarter_end_date and f.user_id = a.user_id and f.account_id = a.account_id
left outer join account_targets as t
  on t.fiscal_year = d.fy and t.fiscal_quarter_number = d.q and t.user_id = a.user_id and t.account_id = a.account_id

In [0]:
CREATE OR REFRESH MATERIALIZED VIEW account_union_individuals AS
select
  'Org Forecast' as view_name
  , cast(NULL as string) as account_id
  , 'All' as account_name
  , f.user_id, f.ae_email, f.bu_plus_2_lead, f.bu_plus_3_lead, f.bu_plus_1_lead_email, f.bu_plus_2_lead_email, f.bu_plus_3_lead_email, f.sales_level, f.user_name
  , f.Business_Unit, f.Region_Level_1, f.Region_Level_2, f.Region_Level_3
  , f.fy, f.q, f.fq, f.fiscal_quarter_start_date    
  , f.dbu_actuals
  , f.dbu_dollars_t7d_avg, f.dbu_dollars_t28d_avg, f.dbu_dollars_t7d_avg_prev, f.dbu_dollars_t28d_avg_prev
  , f.fin_target, f.submitted_my_call, f.submitted_direct_field_consumption_forecast, f.current_ds_forecast, f.submitted_weighted_projection
  , f.forecast_change_pct
  , f.dbu_actuals_coalesced, f.dbu_actuals_or_forecast
  , f.dbu_dollars_t7d_avg_latest, f.dbu_dollars_t7d_adj
  , f.dbu_dollars_t28d_avg_latest, f.dbu_dollars_t28d_adj
  , f.dbu_dollars_t7d_avg_prev_latest, f.dbu_dollars_t7d_prev_adj
  , f.dbu_dollars_t28d_avg_prev_latest, f.dbu_dollars_t28d_prev_adj
  , f.dbu_actuals_current_quarter, f.t7d_proj_left_in_quarter, f.t28d_proj_left_in_quarter, f.target_t7d
  , coalesce(p.last_closed_q, 0) as last_closed_q
  , coalesce(p.days_left_in_quarter, 0) as days_left_in_quarter
  , coalesce(p.is_quarter_closed, false) as is_quarter_closed
  , coalesce(p.is_current_fiscal_quarter, false) as is_current_fiscal_quarter
  , coalesce(p.quarterly_incremental_dbus, 0) as quarterly_incremental_dbus
  , coalesce(p.dbus_in_pipeline_green, 0) as dbus_in_pipeline_green
  , coalesce(p.dbus_in_pipeline_yellow, 0) as dbus_in_pipeline_yellow
  , coalesce(p.dbus_in_pipeline_red, 0) as dbus_in_pipeline_red
  , coalesce(p.dbus_in_pipeline_unknown, 0) as dbus_in_pipeline_unknown
  , coalesce(p.last_day_of_prev_quarter_dbus, 0) as last_day_of_prev_quarter_dbus
from target_forecast_actuals as f
left outer join quarterly_projection as p
  on p.fy = f.fy and p.q = f.q and p.ae_email = f.ae_email

UNION ALL

select
  'Account Forecast' as view_name
  , f.account_id, f.account_name
  , f.user_id, f.ae_email, f.bu_plus_2_lead, f.bu_plus_3_lead, f.bu_plus_1_lead_email, f.bu_plus_2_lead_email, f.bu_plus_3_lead_email, f.sales_level, f.user_name
  , f.Business_Unit, f.Region_Level_1, f.Region_Level_2, f.Region_Level_3
  , f.fy, f.q, f.fq, f.fiscal_quarter_start_date    
  , f.dbu_actuals
  , f.dbu_dollars_t7d_avg, f.dbu_dollars_t28d_avg, f.dbu_dollars_t7d_avg_prev, f.dbu_dollars_t28d_avg_prev
  , f.fin_target, f.submitted_my_call, f.submitted_direct_field_consumption_forecast, f.current_ds_forecast, f.submitted_weighted_projection
  , f.forecast_change_pct
  , f.dbu_actuals_coalesced, f.dbu_actuals_or_forecast
  , f.dbu_dollars_t7d_avg_latest, f.dbu_dollars_t7d_adj
  , f.dbu_dollars_t28d_avg_latest, f.dbu_dollars_t28d_adj
  , f.dbu_dollars_t7d_avg_prev_latest, f.dbu_dollars_t7d_prev_adj
  , f.dbu_dollars_t28d_avg_prev_latest, f.dbu_dollars_t28d_prev_adj
  , f.dbu_actuals_current_quarter, f.t7d_proj_left_in_quarter, f.t28d_proj_left_in_quarter, f.target_t7d
  , coalesce(p.last_closed_q, 0) as last_closed_q
  , coalesce(p.days_left_in_quarter, 0) as days_left_in_quarter
  , coalesce(p.is_quarter_closed, false) as is_quarter_closed
  , coalesce(p.is_current_fiscal_quarter, false) as is_current_fiscal_quarter
  , coalesce(p.quarterly_incremental_dbus, 0) as quarterly_incremental_dbus
  , coalesce(p.dbus_in_pipeline_green, 0) as dbus_in_pipeline_green
  , coalesce(p.dbus_in_pipeline_yellow, 0) as dbus_in_pipeline_yellow
  , coalesce(p.dbus_in_pipeline_red, 0) as dbus_in_pipeline_red
  , coalesce(p.dbus_in_pipeline_unknown, 0) as dbus_in_pipeline_unknown
  , coalesce(p.last_day_of_prev_quarter_dbus, 0) as last_day_of_prev_quarter_dbus
from account_target_forecast_actuals as f
left outer join account_quarterly_projection as p
  on p.fy = f.fy and p.q = f.q and p.user_id = f.user_id and p.account_id = f.account_id

In [0]:
CREATE OR REFRESH MATERIALIZED VIEW quarterly_summary AS
SELECT 
u.view_name, u.ae_email, u.account_id, u.account_name, u.sales_level, u.user_name, u.bu_plus_2_lead, u.bu_plus_3_lead, u.bu_plus_1_lead_email, u.bu_plus_2_lead_email, u.bu_plus_3_lead_email, u.Business_Unit, u.Region_Level_1, u.Region_Level_2, u.Region_Level_3, u.fy, u.fq, u.days_left_in_quarter, u.is_quarter_closed , u.is_current_fiscal_quarter
, u.fin_target, u.submitted_my_call 
, u.submitted_direct_field_consumption_forecast, u.current_ds_forecast, u.submitted_weighted_projection
, u.forecast_change_pct
, u.dbu_actuals_coalesced as dbu_actuals, u.dbu_actuals_or_forecast, u.dbu_actuals_current_quarter
, u.dbu_dollars_t7d_adj, u.dbu_dollars_t7d_prev_adj, u.t7d_proj_left_in_quarter, u.target_t7d
, u.dbu_dollars_t28d_adj, u.dbu_dollars_t28d_prev_adj, u.t28d_proj_left_in_quarter

, coalesce(lag(u.dbu_actuals_or_forecast) over (partition by u.user_id, u.account_id order by u.fq asc), 0) as dbu_actuals_or_forecast_prev_quarter
, coalesce(try_divide(u.fin_target - dbu_actuals_or_forecast_prev_quarter, dbu_actuals_or_forecast_prev_quarter), 0) as qoq_target_growth

, u.quarterly_incremental_dbus 
, u.dbus_in_pipeline_green, u.dbus_in_pipeline_yellow, u.dbus_in_pipeline_red, u.dbus_in_pipeline_unknown
, u.dbus_in_pipeline_green * CAST('${green_confidence_pct}' AS DECIMAL(10,4)) as dbus_in_pipeline_green_in_plan
, u.dbus_in_pipeline_yellow * CAST('${yellow_confidence_pct}' AS DECIMAL(10,4)) as dbus_in_pipeline_yellow_in_plan
, u.dbus_in_pipeline_red * CAST('${red_confidence_pct}' AS DECIMAL(10,4)) as dbus_in_pipeline_red_in_plan
, u.dbus_in_pipeline_unknown * CAST('${unknown_confidence_pct}' AS DECIMAL(10,4)) as dbus_in_pipeline_unknown_in_plan

, case when u.is_quarter_closed then 0 when u.is_current_fiscal_quarter then u.t7d_proj_left_in_quarter else dbu_actuals_or_forecast_prev_quarter  end as baseline
, case when u.is_quarter_closed then "Baseline" when u.is_current_fiscal_quarter then "Baseline (T7D Projection + OG)" else "Baseline (previous quarter's forecast) + OG" end as baseline_label

, case when u.is_quarter_closed then 0 else u.dbu_actuals_coalesced + u.t7d_proj_left_in_quarter end as t7d_flat_projection 
, case when u.is_quarter_closed then 0 else u.dbu_actuals_coalesced + u.t28d_proj_left_in_quarter end as t28d_flat_projection

, baseline * CAST('${best_case_qoq_organic_growth}' AS DECIMAL(10,4)) as best_case_proj_with_og_left_in_quarter
, dbus_in_pipeline_green_in_plan + dbus_in_pipeline_yellow_in_plan + dbus_in_pipeline_red_in_plan + dbus_in_pipeline_unknown_in_plan as best_case_pipe_left_in_quarter
, u.dbu_actuals_coalesced + best_case_proj_with_og_left_in_quarter + best_case_pipe_left_in_quarter + CAST('${best_case_adjustments}' AS DECIMAL(10,4)) as best_case_projection

, baseline * CAST('${worst_case_qoq_organic_growth}' AS DECIMAL(10,4)) as worst_case_proj_with_og_left_in_quarter
, dbus_in_pipeline_green_in_plan as worst_case_pipe_left_in_quarter
, u.dbu_actuals_coalesced + worst_case_proj_with_og_left_in_quarter + worst_case_pipe_left_in_quarter + CAST('${worst_case_adjustments}' AS DECIMAL(10,4)) as worst_case_projection

, u.submitted_my_call - u.fin_target as gap_my_call
, best_case_projection - u.fin_target as gap_best_case
, worst_case_projection - u.fin_target as gap_worst_case
, t7d_flat_projection - u.fin_target as gap_t7d_flat_projection
, t28d_flat_projection - u.fin_target as gap_t28d_flat_projection
, u.submitted_weighted_projection - u.fin_target as gap_weighted_projection
, u.current_ds_forecast - u.fin_target as gap_ds_forecast
, u.submitted_direct_field_consumption_forecast - u.fin_target as gap_directs_forecast
, u.submitted_my_call - u.submitted_direct_field_consumption_forecast as gap_my_call_vs_directs

, u.submitted_my_call - dbu_actuals_or_forecast_prev_quarter as qoq_delta_target 
, best_case_projection - dbu_actuals_or_forecast_prev_quarter as qoq_delta_best_case
, worst_case_projection - dbu_actuals_or_forecast_prev_quarter as qoq_delta_worst_case
, t7d_flat_projection - dbu_actuals_or_forecast_prev_quarter as qoq_delta_t7d_flat_projection
, t28d_flat_projection - dbu_actuals_or_forecast_prev_quarter as qoq_delta_t28d_flat_projection
, u.submitted_weighted_projection - dbu_actuals_or_forecast_prev_quarter as qoq_delta_weighted_projection
, u.current_ds_forecast - dbu_actuals_or_forecast_prev_quarter as qoq_delta_ds_forecast
, u.submitted_direct_field_consumption_forecast - dbu_actuals_or_forecast_prev_quarter as qoq_delta_directs_forecast

, coalesce(try_divide(u.submitted_my_call, u.fin_target), 0) as my_call_att
, coalesce(try_divide(qoq_delta_target, dbu_actuals_or_forecast_prev_quarter), 0) as my_call_qoq_perc
, coalesce(try_divide(u.submitted_direct_field_consumption_forecast, u.fin_target), 0) as directs_att
, coalesce(try_divide(qoq_delta_directs_forecast, dbu_actuals_or_forecast_prev_quarter), 0) as directs_qoq_perc
, coalesce(u.dbu_dollars_t7d_adj - u.dbu_dollars_t7d_prev_adj, 0) as t7d_change_dollar_dbu
, coalesce(u.dbu_dollars_t28d_adj - u.dbu_dollars_t28d_prev_adj, 0) as t28d_change_dollar_dbu
, coalesce(try_divide(t7d_change_dollar_dbu, u.dbu_dollars_t7d_prev_adj), 0) as t7d_change_perc
, coalesce(try_divide(t28d_change_dollar_dbu, u.dbu_dollars_t28d_prev_adj), 0) as t28d_change_perc

, format_number(try_divide(u.submitted_my_call, u.fin_target), '#.#%') || 
  " | " || format_number(try_divide(qoq_delta_target, dbu_actuals_or_forecast_prev_quarter), '#.#%') || 
  " | " || format_number(gap_my_call, '$,###.#') as my_call_text
, format_number(try_divide(u.submitted_direct_field_consumption_forecast, u.fin_target), '#.#%') ||
  " | " || format_number(try_divide(qoq_delta_directs_forecast, dbu_actuals_or_forecast_prev_quarter), '#.#%') ||
  " | " || format_number(gap_directs_forecast, '$,###.#') as directs_fct_text
, format_number(try_divide(gap_best_case, u.fin_target), '#.#%') || 
  " | " || format_number(try_divide(qoq_delta_best_case, dbu_actuals_or_forecast_prev_quarter), '#.#%') || 
  " | " || format_number(gap_best_case, '$,###.#') as best_case_text
, format_number(try_divide(worst_case_projection, u.fin_target), '#.#%') || 
  " | " || format_number(try_divide(qoq_delta_worst_case, dbu_actuals_or_forecast_prev_quarter), '#.#%') || 
  " | " || format_number(gap_worst_case, '$,###.#') as worst_case_text
, format_number(try_divide(submitted_weighted_projection, u.fin_target), '#.#%') || 
  " | " || format_number(try_divide(qoq_delta_weighted_projection, dbu_actuals_or_forecast_prev_quarter), '#.#%') || 
  " | " || format_number(gap_weighted_projection, '$,###.#') as weighted_proj_text
, format_number(try_divide(current_ds_forecast, u.fin_target), '#.#%') || 
  " | " || format_number(try_divide(qoq_delta_ds_forecast, dbu_actuals_or_forecast_prev_quarter), '#.#%') || 
  " | " || format_number(gap_ds_forecast, '$,###.#') as ds_forecast_text
, format_number(try_divide(t7d_flat_projection, u.fin_target), '#.#%') || 
  " | " || format_number(try_divide(qoq_delta_t7d_flat_projection, dbu_actuals_or_forecast_prev_quarter), '#.#%') || 
  " | " || format_number(gap_t7d_flat_projection, '$,###.#') as t7d_proj_text
, format_number(try_divide(t28d_flat_projection, u.fin_target), '#.#%') || 
  " | " || format_number(try_divide(qoq_delta_t28d_flat_projection, dbu_actuals_or_forecast_prev_quarter), '#.#%') || 
  " | " || format_number(gap_t28d_flat_projection, '$,###.#') as t28d_proj_text

from account_union_individuals as u

In [0]:
CREATE OR REFRESH MATERIALIZED VIEW org_forecast AS
SELECT * FROM quarterly_summary
WHERE view_name = 'Org Forecast'

In [0]:
CREATE OR REFRESH MATERIALIZED VIEW account_forecast AS
SELECT * FROM quarterly_summary
WHERE view_name = 'Account Forecast'

In [0]:
-- NOTE: monthly_projection_by_usecase, usecases_filtered and asq_summary are now
-- produced by THIS pipeline (unqualified references => intra-pipeline dependencies).
-- Previously these were cross-schema reads that required the pipeline to run before
-- the separate refresh_agg_mvs job.
CREATE OR REFRESH MATERIALIZED VIEW agg_ucos
CLUSTER BY (Business_Unit, sales_subregion_level_1, sales_subregion_level_2)
AS
WITH monthly_projection_pivoted AS (
  SELECT usecase_id,
    COALESCE(m_2025_02,0) AS m_2025_02, COALESCE(m_2025_03,0) AS m_2025_03, COALESCE(m_2025_04,0) AS m_2025_04,
    COALESCE(m_2025_05,0) AS m_2025_05, COALESCE(m_2025_06,0) AS m_2025_06, COALESCE(m_2025_07,0) AS m_2025_07,
    COALESCE(m_2025_08,0) AS m_2025_08, COALESCE(m_2025_09,0) AS m_2025_09, COALESCE(m_2025_10,0) AS m_2025_10,
    COALESCE(m_2025_11,0) AS m_2025_11, COALESCE(m_2025_12,0) AS m_2025_12, COALESCE(m_2026_01,0) AS m_2026_01,
    COALESCE(m_2026_02,0) AS m_2026_02, COALESCE(m_2026_03,0) AS m_2026_03, COALESCE(m_2026_04,0) AS m_2026_04,
    COALESCE(m_2026_05,0) AS m_2026_05, COALESCE(m_2026_06,0) AS m_2026_06, COALESCE(m_2026_07,0) AS m_2026_07,
    COALESCE(m_2026_08,0) AS m_2026_08, COALESCE(m_2026_09,0) AS m_2026_09, COALESCE(m_2026_10,0) AS m_2026_10,
    COALESCE(m_2026_11,0) AS m_2026_11, COALESCE(m_2026_12,0) AS m_2026_12, COALESCE(m_2027_01,0) AS m_2027_01
  FROM (
    SELECT usecase_id, m, ramping_dbus FROM monthly_projection_by_usecase
  )
  PIVOT (SUM(ramping_dbus) AS dbus FOR m IN (
    '2025-02-01' AS m_2025_02, '2025-03-01' AS m_2025_03, '2025-04-01' AS m_2025_04, '2025-05-01' AS m_2025_05,
    '2025-06-01' AS m_2025_06, '2025-07-01' AS m_2025_07, '2025-08-01' AS m_2025_08, '2025-09-01' AS m_2025_09,
    '2025-10-01' AS m_2025_10, '2025-11-01' AS m_2025_11, '2025-12-01' AS m_2025_12, '2026-01-01' AS m_2026_01,
    '2026-02-01' AS m_2026_02, '2026-03-01' AS m_2026_03, '2026-04-01' AS m_2026_04, '2026-05-01' AS m_2026_05,
    '2026-06-01' AS m_2026_06, '2026-07-01' AS m_2026_07, '2026-08-01' AS m_2026_08, '2026-09-01' AS m_2026_09,
    '2026-10-01' AS m_2026_10, '2026-11-01' AS m_2026_11, '2026-12-01' AS m_2026_12, '2027-01-01' AS m_2027_01
  ))
),
forecast_quarterly_projection_pivoted AS (
  SELECT usecase_id,
    COALESCE(FY26_Q1,0) FY26_Q1, COALESCE(FY26_Q2,0) FY26_Q2, COALESCE(FY26_Q3,0) FY26_Q3, COALESCE(FY26_Q4,0) FY26_Q4,
    COALESCE(FY27_Q1,0) FY27_Q1, COALESCE(FY27_Q2,0) FY27_Q2, COALESCE(FY27_Q3,0) FY27_Q3, COALESCE(FY27_Q4,0) FY27_Q4
  FROM (
    SELECT usecase_id, fq, SUM(quarterly_incremental_dbus) AS quarterly_incremental_dbus
    FROM monthly_projection_by_usecase
    GROUP BY usecase_id, fq
  )
  PIVOT (SUM(quarterly_incremental_dbus) AS dbus FOR fq IN (
    'FY26-Q1' AS FY26_Q1, 'FY26-Q2' AS FY26_Q2, 'FY26-Q3' AS FY26_Q3, 'FY26-Q4' AS FY26_Q4,
    'FY27-Q1' AS FY27_Q1, 'FY27-Q2' AS FY27_Q2, 'FY27-Q3' AS FY27_Q3, 'FY27-Q4' AS FY27_Q4
  ))
),
usecases_filtered_dedup AS (
  SELECT *
  FROM usecases_filtered
  QUALIFY ROW_NUMBER() OVER (PARTITION BY usecase_id, ae_email ORDER BY user_id) = 1
)
SELECT
  c.Business_Unit,
  b.ae_email,
  c.sales_subregion_level_1, c.sales_subregion_level_2, c.sales_subregion_level_3,
  c.account_name, c.deployable_account_name, c.account_executive, c.solution_architect,
  b.dsa_user_name AS dsa, c.arr_band, c.usecase_id, c.usecase_name,
  c.target_onboarding_date, b.target_onboarding_date_fq, c.target_live_date, b.target_live_date_fq,
  c.target_cloud, c.is_migration_usecase,
  COALESCE(NULLIF(TRIM(c.migration_source_platform), ''), 'N/A') AS migration_source_platform_adj,
  c.is_incremental, b.is_keytechwin,
  c.stage, c.stage_number, c.stage_name_ui, c.days_in_stage, c.days_in_validating, c.days_in_scoping,
  c.days_in_evaluating, c.days_in_confirming, c.days_in_onboarding,
  c.usecase_description, c.demand_plan_next_steps, c.implementation_notes,
  b.implementation_partner_name, c.has_ps_project, c.use_case_product_enriched,
  b.estimated_monthly_dollar_dbus, c.estimated_monthly_dollar_dbus_weighted, b.total_ramping_days,
  c.estimated_quarterly_dollar_dbus, c.estimated_quarterly_dollar_dbus_weighted,
  b.planned_techwin_date, b.next_steps_last_updated_date, b.next_steps_stale, b.days_in_stage_bucket, b.usecase_url, b.eval_path, b.eval_doc_link, b.onboarding_doc_link, b.implementation_status,
  b.num_of_blockers, b.blocked_count, b.friction_count, b.blocker_details,
  b.stage_advanced, b.stage_change_description, b.target_date_pulled_foreward, b.target_date_change_description,
  b.target_live_date_diff_days, b.amount_increased, b.amount_change_description, b.change_amount,
  b.change_type_label, b.manager_notes,
  qp.FY26_Q1, qp.FY26_Q2, qp.FY26_Q3, qp.FY26_Q4, qp.FY27_Q1, qp.FY27_Q2, qp.FY27_Q3, qp.FY27_Q4,
  mp.m_2025_02, mp.m_2025_03, mp.m_2025_04, mp.m_2025_05, mp.m_2025_06, mp.m_2025_07, mp.m_2025_08,
  mp.m_2025_09, mp.m_2025_10, mp.m_2025_11, mp.m_2025_12,
  mp.m_2026_01, mp.m_2026_02, mp.m_2026_03, mp.m_2026_04, mp.m_2026_05, mp.m_2026_06, mp.m_2026_07,
  mp.m_2026_08, mp.m_2026_09, mp.m_2026_10, mp.m_2026_11, mp.m_2026_12, mp.m_2027_01,
  b.days_to_go_live, b.days_to_onboarding,
  b.hygiene_rules, b.has_hygiene_issues, b.slippage_risk, b.has_onboarding_slippage_risk,
  b.has_go_live_slippage_risk, b.onboarding_slippage_details, b.go_live_slippage_details,
  b.stage_advanced_count, b.stage_regressed_count, b.live_date_advanced_count, b.live_date_regressed_count,
  b.amount_grew_count, b.amount_shrank_count,
  COALESCE(asq.ASQ_Summary_HTML, 'N/A') AS ASQ_Summary_HTML
FROM main.gtm_silver.use_case_detail AS c
INNER JOIN usecases_filtered_dedup AS b ON b.usecase_id = c.usecase_id
LEFT JOIN forecast_quarterly_projection_pivoted AS qp ON qp.usecase_id = c.usecase_id
LEFT JOIN monthly_projection_pivoted AS mp ON mp.usecase_id = c.usecase_id
LEFT JOIN asq_summary AS asq ON asq.usecase_id = c.usecase_id
WHERE c.Business_Unit = '${business_unit}'
  AND c.sales_subregion_level_1 = '${region_level_1}'